Importing the Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score , precision_score , recall_score , precision_recall_curve , confusion_matrix , auc , classification_report 
import xgboost as xgb
import optuna

Loading the data

In [2]:
X_train_scaled = pd.read_parquet('../data/processed/X_train_scaled.parquet')
X_val_scaled   = pd.read_parquet('../data/processed/X_val_scaled.parquet')
X_test_scaled  = pd.read_parquet('../data/processed/X_test_scaled.parquet')

X_train = pd.read_parquet('../data/processed/X_train.parquet')
X_val   = pd.read_parquet('../data/processed/X_val.parquet')
X_test  = pd.read_parquet('../data/processed/X_test.parquet')

y_train = pd.read_parquet('../data/processed/y_train.parquet')['isFraud']
y_val   = pd.read_parquet('../data/processed/y_val.parquet')['isFraud']
y_test  = pd.read_parquet('../data/processed/y_test.parquet')['isFraud']

Creating an Evaluator Helper Function

In [11]:
# Performing full evaluation of the model , returning metrics deictionary and printing a clean summary

def evaluate_model(model , X , y , model_name , threshold= 0.5):
    proba = model.predict_proba(X)[ : , 1]   #returns 2D array with probabilty of fraud/not fraud ,[ : , 1] we only take fraud probabilty
    preds = (proba >= threshold).astype(int)        #converts probabilities to 0/1 predictions at your chosen threshold:

    # Metrics
    precision = precision_score(y , preds , zero_division=0)
    recall    = recall_score(y, preds, zero_division=0)
    f1        = f1_score(y, preds, zero_division=0)

    pr_precision , pr_recall , _ = precision_recall_curve(y , proba)
    pr_auc = auc(pr_recall, pr_precision) 

    cm = confusion_matrix(y, preds)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn)                  # fraction of legitimate transactions we wrongly blocked

    print(f"Model: {model_name} , Threshold: {threshold}")
    print(f"Precision Recall AUC: {pr_auc:.4f}")
    print(f"F1 score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"FPR:       {fpr:.4f}")
    print(f"{fpr*100:.2f}% legitimate transactions blocked")
    print(f"\nConfusion Matrix:")
    print(f"  TP: {tp:,}  FP: {fp:,}")
    print(f"  FN: {fn:,}  TN: {tn:,}")

    return {
        'model_name': model_name,
        'threshold': threshold,
        'pr_auc': pr_auc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'fpr': fpr,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn
    }

XGBoost (without hyperparameter tuning)